In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

# BB84 Quantum Key Distribution — No Attacker

This notebook simulates the BB84 protocol (Bennett & Brassard, 1984) between Alice and Bob with **no eavesdropper present**.

## Protocol overview
1. **Alice** generates random bits and random basis choices using quantum measurement.
2. **Alice** encodes each bit as a qubit using her chosen basis and sends it to Bob.
3. **Bob** independently chooses a random measurement basis for each qubit and measures it.
4. **Alice and Bob** publicly compare *bases only* (never the bits themselves).
5. They keep only the bits where their bases matched — this is the **shared secret key**.

All random choices are generated by measuring the quantum state |+⟩ = (1/√2)(|0⟩ + |1⟩), which collapses to 0 or 1 with equal probability — true quantum randomness.

In [ ]:
# Setup
simulator = BasicSimulator()

# Number of qubits Alice will send
N = 20

print(f"Simulating BB84 with N={N} qubits.")

Simulating BB84 with N=100 qubits.


### Quantum Random Number Generation

Below function is generated via preparing *n* qubits in the state `|+⟩ = (1/√2)(|0⟩ + |1⟩)` and measuring each one. Each measurement collapses to `|0>` or `|1>` with probability of 1/2

In [ ]:
def quantum_random_bit():
  """
  Generate one truly random bit by preparing |+⟩ = (|0⟩+|1⟩)/√2
  and measuring it. The circuit is always just 1 qubit, so it
  never hits the simulator's qubit limit.
  """
  qc = QuantumCircuit(1, 1)
  qc.h(0)            # |0⟩ → |+⟩
  qc.measure(0, 0)
  compiled = transpile(qc, simulator)
  result = simulator.run(compiled, shots=1).result()
  return int(list(result.get_counts().keys())[0])

def quantum_random_bits(n):
  """Generate n random bits by calling quantum_random_bit() n times."""
  return [quantum_random_bit() for _ in range(n)]

# Quick sanity check
sample = quantum_random_bits(10)
print("Sample quantum random bits:", sample)

Sample quantum random bits: [1, 1, 0, 0, 1, 0, 0, 0, 0, 1]


### Alice Part

1. Picks N random bits (the raw data she wants to share).
2. Picks N random basis choices (0 = standard, 1 = diagonal).
3. Encodes each bit as a 1-qubit state and sends it to Bob.

### Encoding rule:
- Standard basis  (basis=0):  0 → |0⟩,  1 → |1⟩
- Diagonal basis  (basis=1):  0 → |+⟩,  1 → |−⟩

In [ ]:
alice_bits  = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)

def alice_prepare_qubit(bit, basis):
  """
  Alice prepares a 1-qubit circuit encoding `bit` in `basis`.

  Standard (basis=0):  bit 0 → |0⟩ (no gates)
                        bit 1 → |1⟩ (X gate)
  Diagonal (basis=1):  bit 0 → |+⟩ (H gate)
                        bit 1 → |−⟩ (X then H)
  """
  qc = QuantumCircuit(1, 1)
  if bit == 1:
      qc.x(0)
  if basis == 1:
      qc.h(0)
  return qc

# Alice prepares all qubits and places them on the quantum channel
quantum_channel = [alice_prepare_qubit(alice_bits[i], alice_bases[i]) for i in range(N)]

print("=== ALICE ===")
print(f"Bits:   {alice_bits}")
print(f"Bases:  {['S' if b==0 else 'D' for b in alice_bases]}")
print(f"\nAlice has prepared {N} qubits and sent them to Bob.")

=== ALICE ===
Bits:   [1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1]
Bases:  ['D', 'S', 'S', 'S', 'S', 'S', 'S', 'D', 'D', 'D', 'S', 'D', 'D', 'S', 'D', 'D', 'S', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'S', 'D', 'S', 'S', 'D', 'S', 'S', 'D', 'D', 'S', 'S', 'D', 'S', 'S', 'S', 'S', 'S', 'D', 'S', 'D', 'D', 'S', 'S', 'D', 'D', 'S', 'D', 'S', 'S', 'S', 'D', 'S', 'D', 'D', 'S', 'D', 'S', 'D', 'S', 'D', 'D', 'S', 'S', 'S', 'D', 'D', 'D', 'D', 'S', 'D', 'D', 'D', 'S', 'D', 'D', 'D', 'S', 'D', 'D', 'S', 'S', 'D', 'S', 'D', 'S', 'S', 'S', 'S', 'S', 'S', 'D', 'S', 'D']

Alice has prepared 100 qubits and sent them to Bob.


### Bob Part

From lecture, Bob receives qubits from the channel and for each qubit he independently picks a random measurement basis.

For Measurement rule:
- Standard basis  (basis=0): measure directly.
- Diagonal basis  (basis=1): apply H first, then measure.

  (H maps |+⟩→|0⟩ and |−⟩→|1⟩, so the standard measurement now reveals which diagonal state we had.)

If Bob picks the **SAME** basis as Alice: he always recovers Alice's bit.

If he picks the **WRONG** basis: he gets a uniformly random result.

In [ ]:
bob_bases = quantum_random_bits(N)

def bob_measure_qubit(qubit_circuit, basis):
  """
  Bob measures `qubit_circuit` in `basis`.

  Standard (basis=0): measure in the {|0⟩, |1⟩} basis.
  Diagonal (basis=1): apply H then measure (equivalent to
                      measuring in the {|+⟩, |−⟩} basis).
  Returns the measured bit (0 or 1).
  """
  qc = qubit_circuit.copy()
  if basis == 1:
      qc.h(0)
  qc.measure(0, 0)
  compiled = transpile(qc, simulator)
  result = simulator.run(compiled, shots=1).result()
  return int(list(result.get_counts().keys())[0])

bob_bits = [bob_measure_qubit(quantum_channel[i], bob_bases[i]) for i in range(N)]

print("=== BOB ===")
print(f"Bases:  {['S' if b==0 else 'D' for b in bob_bases]}")
print(f"Bits:   {bob_bits}")

=== BOB ===
Bases:  ['S', 'S', 'S', 'D', 'D', 'S', 'D', 'S', 'S', 'D', 'S', 'S', 'D', 'S', 'S', 'D', 'D', 'S', 'S', 'S', 'D', 'S', 'D', 'S', 'S', 'D', 'D', 'S', 'S', 'S', 'S', 'S', 'D', 'D', 'S', 'D', 'S', 'S', 'D', 'D', 'S', 'D', 'D', 'S', 'D', 'S', 'D', 'S', 'D', 'S', 'S', 'D', 'S', 'D', 'S', 'S', 'S', 'S', 'S', 'S', 'D', 'S', 'D', 'S', 'S', 'S', 'S', 'S', 'D', 'S', 'S', 'S', 'D', 'D', 'D', 'S', 'D', 'D', 'S', 'D', 'S', 'S', 'S', 'D', 'S', 'D', 'S', 'S', 'S', 'S', 'D', 'S', 'S', 'S', 'D', 'S', 'S', 'D', 'D', 'S']
Bits:   [0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0]


### Public Channel: Basic Reconciliation

Alice and Bob will annouce their **basis** choices over a public channel. Their bit values are not revealed and only position where same basis is used are kept.

In [ ]:
matching_indices = [i for i in range(N) if alice_bases[i] == bob_bases[i]]

print("=== PUBLIC CHANNEL: BASIS COMPARISON ===")
print(f"Total qubits sent : {N}")
print(f"Matching positions: {len(matching_indices)}  (expected ~{N//2})")
print(f"Discarded positions: {N - len(matching_indices)}")

=== PUBLIC CHANNEL: BASIS COMPARISON ===
Total qubits sent : 100
Matching positions: 56  (expected ~50)
Discarded positions: 44


### Key Extraction

Below shows how Alice and Bob can extract their shared key from matching positions. Since both of them uses the same basis, they should have identical bit values at matching positions.

In [ ]:
alice_key = [alice_bits[i] for i in matching_indices]
bob_key   = [bob_bits[i]   for i in matching_indices]

print("=== SHARED SECRET KEY ===")
print(f"Alice's key : {alice_key}")
print(f"Bob's key   : {bob_key}")
print()

if alice_key == bob_key:
  print(f"Keys MATCH! Shared key length: {len(alice_key)} bits.")
  print("Alice and Bob now share a secret key they can use as a one-time pad.")
else:
  # Count positions where Alice and Bob's bits differ (True=1, False=0).
  mismatches = sum(a != b for a, b in zip(alice_key, bob_key))
  print(f"Keys do NOT match ({mismatches} mismatches) — this should not happen without an attacker.")

=== SHARED SECRET KEY ===
Alice's key : [0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1]
Bob's key   : [0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1]

Keys MATCH! Shared key length: 56 bits.
Alice and Bob now share a secret key they can use as a one-time pad.


### Summary

In [ ]:
print("===== BB84 PROTOCOL SUMMARY (No Attacker) =====")
print(f"  Qubits sent by Alice    : {N}")
print(f"  Basis matches           : {len(matching_indices)} (~{100*len(matching_indices)/N:.0f}%)")
print(f"  Final key length        : {len(alice_key)} bits")
print(f"  Keys agree              : {alice_key == bob_key}")
print()
print("Protocol complete. The shared key can be used as a one-time pad.")

===== BB84 PROTOCOL SUMMARY (No Attacker) =====
  Qubits sent by Alice    : 100
  Basis matches           : 56 (~56%)
  Final key length        : 56 bits
  Keys agree              : True

Protocol complete. The shared key can be used as a one-time pad.
